In [1]:
%cd ../

/nas/zhangtianning.di/projects/unique_data_build


In [ ]:
import jsonlines

In [3]:
# all in one
import os
import json

from tqdm.auto import tqdm, trange
from nameparser import HumanName
from elasticsearch import Elasticsearch
from elasticsearch.helpers import streaming_bulk
from collections.abc import MutableMapping
import logging


In [4]:

with open("/nvme/zhangtianning.di/semantic_scholar/vene_alias_map.json", 'r') as f:
    vene_name_to_alias = json.load(f)

In [4]:
from python_script.reference_reterive.Reference import *

In [4]:
ES_INDEX="integrate20240311"
es = Elasticsearch("http://10.140.52.119:9200")
es.indices.refresh()
# if es.indices.exists(index=ES_INDEX):
#     es.indices.delete(index=ES_INDEX)
#es.indices.refresh()

ObjectApiResponse({'_shards': {'total': 0, 'successful': 0, 'failed': 0}})

In [12]:
indices = es.indices.get_alias()

# Print the names of all indices
for index in indices:
    print(index)

In [7]:
from elasticsearch import Elasticsearch
# Get all the repositories
repositories = es.snapshot.get_repository()
# Iterate over each repository
for repo_name, repo_info in repositories.items():
    print(f"Repository: {repo_name}")
    print(f"Repository information: {repo_info}")

    # Get the list of snapshots in the repository
    snapshots = es.snapshot.get(repository=repo_name, snapshot="_all")
    
    # Print the snapshot information
    print("Snapshots:")
    for snapshot in snapshots["snapshots"]:
        print(f"  Snapshot: {snapshot['snapshot']}")
        print(f"  State: {snapshot['state']}")
        print(f"  Start Time: {snapshot['start_time']}")
        print(f"  End Time: {snapshot['end_time']}")
        print(f"  Indices: {snapshot['indices']}")
        print("  -------------------------")

    print("===========================")
    

Repository: digital_resource
Repository information: {'type': 'fs', 'settings': {'compress': 'true', 'location': '/var/lib/elasticsearch/repo'}}
Snapshots:


In [17]:
#es.snapshot.delete(repository='digital_resource', snapshot='paper:crf+ss+arxiv')

In [95]:
# from elasticsearch import Elasticsearch
# ES_INDEX="integrate20240311"
# es = Elasticsearch("http://10.140.52.118:9200")
# es.indices.refresh()

ObjectApiResponse({'_shards': {'total': 2, 'successful': 1, 'failed': 0}})

In [15]:
# Configuration
repository_name           = 'digital_resource'
snapshot_name             = '20240320:crf+ss+arxiv'
snapshot_repo_location    = "/var/lib/elasticsearch/repo"
indices_to_snapshot       = ES_INDEX
for key,val in es.snapshot.get(repository=repository_name, snapshot=snapshot_name)['snapshots'][0].items():
    print(f"{key}->{val}")

IndexError: list index out of range

In [22]:
# Create the Elasticsearch client instances for the source and destination
source_es = es
# Register a Snapshot Repository on the source
source_es.snapshot.create_repository(
    name=repository_name,
    body={
        "type": "fs",
        "settings": {
            "location": snapshot_repo_location,
            "compress": True
        }
    }
)

ObjectApiResponse({'acknowledged': True})

In [25]:
source_es.snapshot.create(
    repository=repository_name,
    snapshot=snapshot_name,
    body={
        "indices": indices_to_snapshot,
        "ignore_unavailable": True,
        "include_global_state": False
    },
    wait_for_completion=True  # Block until the snapshot is complete
)

ConnectionTimeout: Connection timed out

TypeError: SnapshotClient.delete() missing 1 required keyword-only argument: 'snapshot'

In [ ]:
curl -X DELETE localhost:9200/_snapshot/my_backup/snapshot_1

Repository: my_backup
Repository information: {'type': 'fs', 'uuid': 'A6o5kiqNRv-zRwm8Hkb8CA', 'settings': {'compress': 'true', 'location': '/var/lib/elasticsearch/repo'}}
Snapshots:
  Snapshot: paper:crf+ss+arxiv
  State: SUCCESS
  Start Time: 2024-03-14T08:15:59.577Z
  End Time: 2024-03-14T09:23:22.022Z
  Indices: ['integrate20240311']
  -------------------------
Repository: digital_resource
Repository information: {'type': 'fs', 'uuid': 'A6o5kiqNRv-zRwm8Hkb8CA', 'settings': {'compress': 'true', 'location': '/var/lib/elasticsearch/repo'}}
Snapshots:
  Snapshot: paper:crf+ss+arxiv
  State: SUCCESS
  Start Time: 2024-03-14T08:15:59.577Z
  End Time: 2024-03-14T09:23:22.022Z
  Indices: ['integrate20240311']
  -------------------------


In [6]:
es = Elasticsearch(["http://10.140.52.119:9200"])
for key,val in es.snapshot.get(repository='digital_resource', snapshot='paper:crf+ss+arxiv')['snapshots'][0].items():
    print(f"{key}->{val}")

NotFoundError: NotFoundError(404, 'snapshot_missing_exception', '[digital_resource:paper:crf+ss+arxiv] is missing')

In [197]:
source_es.snapshot.create(
    repository=repository_name,
    snapshot=snapshot_name,
    body={
        "indices": indices_to_snapshot,
        "ignore_unavailable": True,
        "include_global_state": False
    },
    wait_for_completion=True  # Block until the snapshot is complete
)


ConnectionTimeout: Connection timed out

##### Load Indentity Pool

In [8]:
import redis
from python_script.build_redis_database.utils import *
redis_host = "localhost"
redis_port = 6379
redis_password = ""
r = redis.StrictRedis(host=redis_host, port=redis_port, password=redis_password, decode_responses=True)

In [9]:
def get_digital_worth_index_from_unique_id(unique_id:UniqueID):
    for alias_type, alias_value in unique_id.to_dict().items():
        digital_worth_index = get_index_by_alias(r,alias_type, alias_value)
        if digital_worth_index is not None:
            return digital_worth_index

## Load data from arxivemetadata

In [74]:
index_part=0
num_parts =2

resource_dir = '/nvme/zhangtianning.di/datasets/whole_arxiv_data/arxiv-metadata-oai-snapshot.json'

totally_paper_num = 2231517
divided_nums = np.linspace(0, totally_paper_num - 1, num_parts)
divided_nums = [int(s) for s in divided_nums]
start_index = divided_nums[index_part]
end_index   = divided_nums[index_part + 1]
conflict_doi_that_should_same_but_splited = []
conflict_doi_that_should_same_but_splited_path = f'/nvme/zhangtianning.di/datasets/whole_arxiv_data/arxiv-metadata_conflict/{index_part}_{num_parts}.json'
conflict_doi_that_should_same_but_splited_dir  = os.path.dirname(conflict_doi_that_should_same_but_splited_path)
os.makedirs(conflict_doi_that_should_same_but_splited_dir, exist_ok=True)

# Connect to Redis
redis_host = "localhost"
redis_port = 6379
redis_password = ""
r = redis.StrictRedis(host=redis_host, port=redis_port, password=redis_password, decode_responses=True)

with open(resource_dir,'r') as f:
    with tqdm(range(end_index-start_index), desc="Main") as pbar:
        for unique_index, line in enumerate(f):
            break

Main:   0%|          | 0/2231516 [00:00<?, ?it/s]

In [76]:
identity_set = json.loads(line)

In [91]:
identity_set

{'id': '0704.0001',
 'submitter': 'Pavel Nadolsky',
 'authors': "C. Bal\\'azs, E. L. Berger, P. M. Nadolsky, C.-P. Yuan",
 'title': 'Calculation of prompt diphoton production cross sections at Tevatron and\n  LHC energies',
 'comments': '37 pages, 15 figures; published version',
 'journal-ref': 'Phys.Rev.D76:013009,2007',
 'doi': '10.1103/PhysRevD.76.013009',
 'report-no': 'ANL-HEP-PR-07-12',
 'categories': 'hep-ph',
 'license': None,
 'abstract': '  A fully differential calculation in perturbative quantum chromodynamics is\npresented for the production of massive photon pairs at hadron colliders. All\nnext-to-leading order perturbative contributions from quark-antiquark,\ngluon-(anti)quark, and gluon-gluon subprocesses are included, as well as\nall-orders resummation of initial-state gluon radiation valid at\nnext-to-next-to-leading logarithmic accuracy. The region of phase space is\nspecified in which the calculation is most reliable. Good agreement is\ndemonstrated with data from th

In [96]:
Reference.load_from_dict(identity_set|{'arxiv':identity_set['id']})

Reference:
unique_id |-> Paper:
   doi |-> 10.1103/physrevd.76.013009
   arxiv |-> 0704.0001
title |-> Calculation of prompt diphoton production cross sections at Tevatron and
  LHC energies
author |-> ["C. Bal\\'azs", ' E. L. Berger', ' P. M. Nadolsky', ' C.-P. Yuan']

## Load data from crossref

In [ ]:
from pathlib import Path
from tqdm.auto import tqdm
import gzip,json

resource_dir = "/nvme/zhangtianning.di/crossref/CrossrefData/"
resource_files= list(Path(resource_dir).glob("*.gz"))

In [61]:
resource_dir = "/nvme/zhangtianning.di/crossref/update_citation_content/failcase/"
resource_files= [str(t) for t in Path(resource_dir).glob("*.jsonl")]

In [63]:
filename = "/nvme/zhangtianning.di/crossref/update_citation_content/filelist.json" #resource_files[1]
with open(filename,'w') as f:
    json.dump(resource_files, f)

In [62]:
resource_files[0]

'/nvme/zhangtianning.di/crossref/update_citation_content/failcase/17701.jsonl'

In [15]:
filename = "/nvme/zhangtianning.di/crossref/CrossrefData/9361.json.gz" #resource_files[1]
with gzip.open(filename, 'r') as f:
    json_data = json.load(f)

In [2]:
resource_files = "/nvme/zhangtianning.di/crossref/CrossrefData/15182.json.gz"

In [10]:
from python_script.CitationStyleLanguage import *
from python_script.reference_reterive.Reference import *

In [27]:
badrowdatas=[]
for rowdata in json_data['items']:
    citation = CitationStyleLanguage.from_dict(rowdata)
    if not citation.is_good_for_citation():
        badrowdatas.append(rowdata)
        continue

In [43]:
rowdata=json.loads("""{"URL": "http://dx.doi.org/10.1037/e380282004-001", "resource": {"primary": {"URL": "http://doi.apa.org/get-pe-doi.cfm?doi=10.1037/e380282004-001"}}, "member": "15", "score": 0.0, "created": {"date-parts": [[2013, 2, 23]], "date-time": "2013-02-23T11:32:01Z", "timestamp": 1361619121000}, "container-title": ["PsycEXTRA Dataset"], "issued": {"date-parts": [[null]]}, "prefix": "10.1037", "reference-count": 0, "indexed": {"date-parts": [[2022, 4, 2]], "date-time": "2022-04-02T01:34:50Z", "timestamp": 1648863290944}, "author": [{"given": "Ellen K.", "family": "Slicker", "sequence": "first", "affiliation": []}, {"given": "Billie K.", "family": "Picklesimer", "sequence": "additional", "affiliation": []}], "DOI": "10.1037/e380282004-001", "is-referenced-by-count": 0, "alternative-id": ["380282004-001"], "subtitle": ["(380282004-001)"], "content-domain": {"domain": [], "crossmark-restriction": false}, "title": ["Parenting Style and Development of Life-Skills in Older Adolescents"], "source": "Crossref", "type": "dataset", "publisher": "American Psychological Association (APA)", "institution": [{"name": "American Psychological Association", "acronym": ["APA"]}], "references-count": 0, "deposited": {"date-parts": [[2013, 2, 23]], "date-time": "2013-02-23T11:52:31Z", "timestamp": 1361620351000}}""")

In [50]:
CitationStyleLanguage.from_dict(rowdata)

CSL:
   author |-> [{'given': 'Ellen K.', 'family': 'Slicker'}, {'given': 'Billie K.', 'family': 'Picklesimer'}]
   title |-> Parenting Style and Development of Life-Skills in Older Adolescents
   type |-> dataset
   container-title |-> PsycEXTRA Dataset

In [14]:
CitationStyleLanguage.from_dict(rowdata)

CSL:
   issued |-> {'date-parts': [[2015, 1, 20]]}
   author |-> [{'given': 'Y. K.', 'family': 'Chan', 'sequence': 'first', 'affiliation': []}, {'given': 'D.', 'family': 'Wong', 'sequence': 'additional', 'affiliation': []}, {'given': 'H. K.', 'family': 'Yeung', 'sequence': 'additional', 'affiliation': []}, {'given': 'P. K.', 'family': 'Man', 'sequence': 'additional', 'affiliation': []}, {'given': 'H. C.', 'family': 'Shum', 'sequence': 'additional', 'affiliation': []}]
   title |-> A Low-Molecular-Weight Oil Cleaner For Removal of Leftover Silicone Oil Intraocular Tamponade
   volume |-> 56
   page |-> 1014-1022
   type |-> journal-article
   container-title |-> Investigative Ophthalmology &amp; Visual Science

In [201]:
from tqdm.auto import tqdm

In [204]:
es.get(index=ES_INDEX, id=digital_index,)['_source']

{'unique_id.doi': '10.1163/2214-8655_lgo_lgo_08_1029',
 'title': 'Lexicon Gregorianumσυνέμπορος, ου, ὁ',
 'author': None,
 'journal_volume': None,
 'journal_page': None,
 'year': None,
 'publisher': 'Brill',
 'content': None,
 'journal': 'lexicon gregorianum online'}

In [202]:
Analysis= {'update':0, 'new_add': 0, 'crossref_normal_skip': 0,'escape_by_too_less_information':[], 'escape_by_no_unique_id':[]}
for i, rowdata in tqdm(enumerate(json_data['items']),total=len(json_data['items'])):
    if 'alternative-id' in rowdata and len(set(rowdata['alternative-id']))>1:
        Analysis['crossref_normal_skip']+=1
        continue
    newref = Reference.load_from_dict(rowdata)
    digital_index = get_digital_worth_index_from_unique_id(newref.unique_id)
    if digital_index is None:
        Analysis['escape_by_no_unique_id'].append(newref.unique_id)
        continue
    if not es.exists(index=ES_INDEX, id=digital_index).body:
        if decide_whether_add_the_ref_into_the_database(newref):
            body = format_es_paper(newref)
            #es.index(index=ES_INDEX, id=digital_index, body=body)
            Analysis['new_add']+=1
            
        else:
            Analysis['escape_by_too_less_information'].append(newref.unique_id.doi)
            
    else:
        body  = es.get(index=ES_INDEX, id=digital_index)['_source']
        if 'doc' in body:
            should_upate = True
            body = body['doc']
            #es.delete(index=ES_INDEX, id=digital_index)
            #es.index(index=ES_INDEX, id=digital_index, body=body)
            
        old_ref = Reference.load_from_dict(body)
        new_information = addtition_information(old_ref, newref)
        if len(new_information) > 0:
            body = body|format_es_paper(new_information)
            #es.update(index=ES_INDEX, id=digital_index, body={'doc':body}) #<-- you must use doc
            Analysis['update']+=1

  0%|          | 0/5000 [00:00<?, ?it/s]

In [193]:
digital_index

'CR.19725.57'

In [150]:
body

{'unique_id.doi': '10.3389/fnins.2018.00812.s008',
 'title': 'Table_1.PDF',
 'author': None,
 'journal_volume': None,
 'journal_page': None,
 'year': None,
 'publisher': 'Frontiers Media SA',
 'content': None,
 'journal': 'frontiers media sa'}

In [147]:
len(escape_doi)

335

In [137]:
body  = es.get(index=ES_INDEX, id=digital_index)['_source']
old_ref = Reference.load_from_dict(body)
new_information = addtition_information(old_ref, newref)

In [141]:
def format_es_paper(ref:Reference|Dict):
    if isinstance(ref,Reference):
        out = ref.to_flatten_dict()
    else:
        out = ref
    if out.get('author',None):
        author_list = out.pop('author')
        for author_order, name in zip(range(3),author_list ):
            out[f'author.{author_order}'] = name
    if out.get('journal',None):
        journal_name =  out.pop('journal')
        journal_name = journal_name.lower()
        if journal_name in vene_name_to_alias:
            journal_name = ", ".join(vene_name_to_alias[journal_name])
        out['journal'] = journal_name
    return out

In [142]:
format_es_paper(new_information)

{'journal_page': '3-3', 'publisher': 'OECD', 'author.0': 'OECD'}

In [135]:
new_information

{'author': ['OECD'], 'journal_page': '3-3', 'publisher': 'OECD'}

In [134]:
addtition_information

<function __main__.addtition_information(old_ref: python_script.reference_reterive.Reference.ReferenceBase, new_ref: python_script.reference_reterive.Reference.ReferenceBase)>

In [133]:
newref

Reference:
unique_id |-> Paper:
   doi |-> 10.1787/9789264306943-1-en
title |-> Foreword
author |-> ['OECD']
journal |-> The Future of Social Protection
journal_page |-> 3-3
year |-> 2018
publisher |-> OECD

In [132]:
def addtition_information(old_ref:ReferenceBase, new_ref:ReferenceBase):
    #### firstly deal with None value
    old_pool = old_ref.to_dict()
    new_pool = new_ref.to_dict()
    total_key = vars(old_ref).keys()
    new_information = {}
    for key in total_key:
        if old_pool.get(key, None) is None and new_pool.get(key, None) is not None:
            new_information[key] = new_pool[key]
        elif old_pool.get(key, None) is not None and new_pool.get(key, None) is not None:
            continue
            if key in ['unique_id']:continue
            a = old_pool[key]
            b = new_pool[key]
            if key == 'journal':
                a = a.lower()
                b = b.lower()
                if b not in a:
                    print(f"{key}: a={a} b={b}")
            else:
                if a !=b :
                    print(f"{key}: a={a} b={b}")

    return new_information

In [40]:
addtition_information(old_ref, newref)

{'publisher': 'Springer Science and Business Media LLC'}

In [17]:
old_ref.addtition_information(newref)

{}

## Load data from semantic_scholar

In [97]:
def format_es_paper(ref:Reference|Dict):
    if isinstance(ref,Reference):
        out = ref.to_flatten_dict()
    else:
        out = ref
    if out.get('author',None):
        author_list = out.pop('author')
        for author_order, name in zip(range(3),author_list ):
            out[f'author.{author_order}'] = name
    if out.get('journal',None):
        journal_name =  out.pop('journal')
        journal_name = journal_name.lower()
        if journal_name in vene_name_to_alias:
            journal_name = ", ".join(vene_name_to_alias[journal_name])
        out['journal'] = journal_name
    return out

def generate_es_filter_stream(input_file_path):
    """Reads the file through csv.DictReader() and for each row
    yields a single document. This function is passed into the bulk()
    helper to create many documents in sequence.
    """
    with jsonlines.open(input_file_path, mode="r") as f:

        for rowdata in f:
            reference = Reference.load_from_dict(rowdata)
            doc = {
                "_id": get_digital_worth_index_from_unique_id(reference.unique_id),
            }|format_es_paper(reference)
            yield doc

In [98]:
import pathlib
import os
all_paths = list(pathlib.Path("/nvme/zhangtianning.di/semantic_scholar/split").glob('20231201*'))


In [99]:
def line_count(file_path):
    return int(os.popen(f'wc -l {file_path}').read().split()[0])

In [102]:
file_idx = 0
file_path= all_paths[0]
actions = generate_es_filter_stream(file_path)
# actions = generate_actions(file_path)
number_of_lengths = line_count(file_path)

In [85]:
with jsonlines.open(f"error_file{file_path.name}.jsonl", 'w') as error_file:
    progress = tqdm(unit="docs", total=number_of_lengths, desc=f"The progress of the {file_idx+1}/{len(all_paths)} file")
    successes = 0
    for ok, action in streaming_bulk(
        client=es, index=ES_INDEX, actions=actions,
        # chunk_size=16,
    ):
        progress.update(1)
        successes += ok
        if not ok:
            error_file.writeline(action)

logging.info(f"Successfully indexed {successes}/{number_of_lengths} documents")
if number_of_lengths != successes:
    logging.info(f"{number_of_lengths-successes} failed files in error_file{file_path.name}.jsonl")
else:
    os.remove(f"error_file{file_path.name}.jsonl")

The progress of the 1/2900 file:   0%|          | 0/72000 [00:00<?, ?docs/s]

In [100]:
from collections import defaultdict
def strfy(x):
    while isinstance(x, list):
        x = " ".join(x)
    return x
def get_es_query_dict_from_Reference(ref:Reference):
    search_should = []
    flags = defaultdict(int)
    if ref.author:
        for author_i in ref.author[:3]:
            search_should.append({
                "multi_match": {"query": "{}".format(author_i),"fields": ["author.0", "author.1", "author.2"],"type": "best_fields"}
            })
            flags['author'] += 1
    
    if ref.journal_volume:
        search_should.append({
            "multi_match": {"query": strfy(ref.journal_volume),"fields": ["journal_volume"],"type": "best_fields"}
        })
        flags['volume'] = 1
    
    if ref.journal_page:
        search_should.append({
            "multi_match": {"query": strfy(ref.journal_page),"fields": ["journal_page"],"type": "best_fields"}
        })
        flags['page'] = 1
    
    if ref.title:
        search_should.append({
            "multi_match": {"query": strfy(ref.title),"fields": ["title" ],"type": "best_fields"}
        })
        flags['title'] = 1
                      
    if ref.journal:
        search_should.append({
            "multi_match": {"query": strfy(ref.journal),"fields": ["journal"],"type": "best_fields"}
        })
        flags['vene'] = 1
    

    if ref.year:
        search_should.append({
            "multi_match": {"query": strfy(ref.year),"fields": ["year"],"type": "best_fields"}
        })
        flags['year'] = 1

    
    ### form the query 
    query_dict = {
        "bool": {
            "should": search_should
        }
    }
    return query_dict

In [128]:
reference.to_dict()

{'unique_id': Paper:
    doi |-> 10.1007/s13555-022-00698-x
    pubmed |-> 35262878
    pmc |-> 9021334,
 'title': 'Onychomycosis: Recommendations for Diagnosis, Assessment of Treatment Efficacy, and Specialist Referral. The CONSONANCE Consensus Project',
 'author': ['B. Piraccini',
  'M. Starace',
  'A. Rubin',
  'N. D. Di Chiacchio',
  'M. Iorizzo',
  'D. Rigopoulos'],
 'journal': 'Dermatology and Therapy',
 'journal_volume': '12',
 'journal_page': '885 - 898',
 'year': 2022,
 'publisher': None,
 'content': None}

In [105]:
CitationStyleLanguage.from_dict(data)

CSL:
   title |-> CuAl1.6 Fe0.4O4/凹凸棒石复合材料的制备及可见光催化性能研究
   year |-> 2014

In [101]:
query = get_es_query_dict_from_Reference(ref)

NameError: name 'ref' is not defined

In [88]:
es.get(index=ES_INDEX, id='44468656')

ObjectApiResponse({'_index': 'integrate20240311', '_id': '44468656', '_version': 1, '_seq_no': 0, '_primary_term': 1, 'found': True, '_source': {'unique_id.mag': '431415137', 'title': 'CuAl1.6 Fe0.4O4/凹凸棒石复合材料的制备及可见光催化性能研究', 'journal': None, 'journal_volume': '33', 'journal_page': '204-208', 'year': 2014, 'publisher': None, 'content': None, 'author.0': '苏广慧', 'author.1': '陈龙', 'author.2': '崔佳萌'}})

In [90]:
ref

Reference:
unique_id |-> Paper:
   doi |-> 10.3348/jkrs.1983.19.1.107
   mag |-> 2605332053
title |-> Clinical and Radiological Evaluation of Subphrenic Abscess
author |-> ['Y. Kim', 'J. Oh', 'D. Hur']
journal |-> Journal of the Korean Society of Radiology
journal_volume |-> 19
journal_page |-> 107-115
year |-> 1983

In [94]:
answer = es.search(index=ES_INDEX, query=query, size=1).body['hits']['hits'][0]['_source']

NotFoundError: NotFoundError(404, 'index_not_found_exception', 'no such index [integrate20240311]', integrate20240311, index_or_alias)

In [92]:
answer

{'unique_id.doi': '10.3348/jkrs.1983.19.1.107',
 'unique_id.mag': '2605332053',
 'title': 'Clinical and Radiological Evaluation of Subphrenic Abscess',
 'journal_volume': '19',
 'journal_page': '107-115',
 'year': 1983,
 'publisher': None,
 'content': None,
 'author.0': 'Y. Kim',
 'author.1': 'J. Oh',
 'author.2': 'D. Hur',
 'journal': 'journal of the korean society of radiology, j korean soc radiol'}

In [73]:
ref

Reference:
unique_id |-> Paper:
   doi |-> 10.3348/jkrs.1983.19.1.107
   mag |-> 2605332053
title |-> Clinical and Radiological Evaluation of Subphrenic Abscess
author |-> ['Y. Kim', 'J. Oh', 'D. Hur']
journal |-> Journal of the Korean Society of Radiology
journal_volume |-> 19
journal_page |-> 107-115
year |-> 1983

In [79]:
answer

{'unique_id.doi': '10.3348/jkrs.1983.19.1.107',
 'unique_id.mag': '2605332053',
 'title': 'Clinical and Radiological Evaluation of Subphrenic Abscess',
 'journal_volume': '19',
 'journal_page': '107-115',
 'year': 1983,
 'publisher': None,
 'content': None,
 'author_0': 'Y. Kim',
 'author_1': 'J. Oh',
 'author_2': 'D. Hur',
 'journal': 'journal of the korean society of radiology, j korean soc radiol'}

In [78]:
answer_ref = Reference.load_from_dict(answer)
answer_ref

Reference:
unique_id |-> Paper:
   doi |-> 10.3348/jkrs.1983.19.1.107
   mag |-> 2605332053
title |-> Clinical and Radiological Evaluation of Subphrenic Abscess
journal |-> journal of the korean society of radiology, j korean soc radiol
journal_volume |-> 19
journal_page |-> 107-115
year |-> 1983

In [47]:
with jsonlines.open(all_paths[0], mode="r") as f:
    for rowdata in f:
        data = rowdata
        break